# Phase 3 — SAM + LoRA Adapter
**Project:** Can LoRA-Adapted SAM Match Specialist Polyp Networks?

This notebook implements the main contribution:
- Load frozen SAM ViT-H image encoder
- Inject LoRA adapters into Q/V attention projections
- Train only LoRA params + lightweight mask decoder
- Compare parameter efficiency vs U-Net baseline
- Evaluate on all 5 splits (seen + unseen)

**Compute needed:** A100 or L4 (Google Colab Pro) for SAM ViT-H.
For quick testing, swap `model_type='vit_b'` to use the smaller backbone on a free T4.

## 1. Setup

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO = '/content/msu2026summer_final_project'
    if not os.path.exists(REPO):
        !git clone --quiet https://github.com/palism1/msu2026summer_final_project.git {REPO}
    %cd {REPO}
    !git fetch --quiet origin && git reset --hard origin/main
    !find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null; true
    !pip install -q -r requirements.txt
    !pip install -q git+https://github.com/facebookresearch/segment-anything.git

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  |  GPU: {torch.cuda.get_device_name(0) if device=="cuda" else "N/A"}')

## 2. Download SAM Checkpoint

SAM checkpoints are released by Meta on GitHub.
ViT-H is the largest/best; ViT-B fits on a free T4 for quick experiments.

In [ ]:
# Choose backbone size
# SAM_TYPE = 'vit_b'   # ~375 MB  — free Colab T4
SAM_TYPE = 'vit_h'     # ~2.4 GB  — Colab A100/L4 (recommended for final results)

CKPT_URLS = {
    'vit_h': ('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
               'sam_vit_h_4b8939.pth'),
    'vit_l': ('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth',
               'sam_vit_l_0b3195.pth'),
    'vit_b': ('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
               'sam_vit_b_01ec64.pth'),
}

url, fname = CKPT_URLS[SAM_TYPE]
if not os.path.exists(fname):
    print(f'Downloading SAM {SAM_TYPE} checkpoint (~{"2.4" if SAM_TYPE=="vit_h" else "375M"} GB)...')
    !wget -q --show-progress {url} -O {fname}
else:
    print(f'Checkpoint already downloaded: {fname}')

## 3. Build SAM-LoRA Model

In [ ]:
from src.models import build_sam_lora

LORA_R     = 4
LORA_ALPHA = 8.0
IMG_SIZE   = 352

model = build_sam_lora(
    sam_checkpoint=fname,
    model_type=SAM_TYPE,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.1,
    img_size=IMG_SIZE,
    device=device,
)
model = model.to(device)

total     = model.total_parameters()
trainable = model.trainable_parameters()
print(f'SAM {SAM_TYPE.upper()} + LoRA (r={LORA_R})')
print(f'  Total parameters:     {total:>12,}')
print(f'  Trainable parameters: {trainable:>12,}  ({100*trainable/total:.2f}%)')
print(f'  Frozen parameters:    {total-trainable:>12,}')

In [ ]:
# Quick forward pass check
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    out = model(dummy)
print(f'Forward pass OK: {list(dummy.shape)} → {list(out.shape)}')

## 4. Parameter Efficiency Comparison

In [ ]:
from src.models import build_unet
from src.models.unet import count_parameters

unet_params = count_parameters(build_unet())

print('Method                   Total params  Trainable params  % trainable')
print('-' * 70)
print(f'U-Net (ResNet-34)      {unet_params["total"]:>13,}   {unet_params["trainable"]:>13,}    {100*unet_params["trainable"]/unet_params["total"]:.1f}%')
print(f'SAM-{SAM_TYPE.upper()} + LoRA (r={LORA_R})  {total:>13,}   {trainable:>13,}    {100*trainable/total:.2f}%')

## 5. Training (Phase 3 — run after Phase 2 baseline)

In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader
from src.data import PolypDataset, build_splits, get_train_transform, get_val_transform
from src.metrics import MetricTracker
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm
import numpy as np

DATA_ROOT = Path('data/polyp')
SEED      = 42
BATCH     = 4    # smaller batch for large SAM encoder
EPOCHS    = 50
LR        = 5e-5
PATIENCE  = 8

splits = build_splits(DATA_ROOT, seed=SEED)

train_ds = PolypDataset(
    splits['train']['image_paths'], splits['train']['mask_paths'],
    transform=get_train_transform(IMG_SIZE))
val_ds = PolypDataset(
    splits['seen_kvasir']['image_paths'] + splits['seen_clinicdb']['image_paths'],
    splits['seen_kvasir']['mask_paths']  + splits['seen_clinicdb']['mask_paths'],
    transform=get_val_transform(IMG_SIZE))

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

def dice_bce_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    bce = nn.functional.binary_cross_entropy_with_logits(logits, targets)
    inter = (probs * targets).sum(dim=(1,2,3))
    dice = 1 - (2*inter + eps)/(probs.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3)) + eps)
    return bce + dice.mean()

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
CKPT_DIR  = Path(f'checkpoints/sam_lora_{SAM_TYPE}/seed{SEED}')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

best_dice, patience_count = 0.0, 0
history = {'train_loss': [], 'val_dice': [], 'val_iou': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f'Epoch {epoch:03d} train', leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = dice_bce_loss(model(imgs), masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    epoch_loss /= len(train_loader)

    model.eval()
    tracker = MetricTracker()
    with torch.no_grad():
        for imgs, masks in val_loader:
            probs = torch.sigmoid(model(imgs.to(device))).cpu().numpy()
            for i in range(len(probs)):
                tracker.update(probs[i,0], masks[i,0].numpy())
    scores = tracker.compute()
    scheduler.step()

    history['train_loss'].append(epoch_loss)
    history['val_dice'].append(scores['dice'])
    history['val_iou'].append(scores['iou'])

    print(f'Epoch {epoch:03d} | loss={epoch_loss:.4f} | val_dice={scores["dice"]:.4f} | val_iou={scores["iou"]:.4f}')

    if scores['dice'] > best_dice:
        best_dice = scores['dice']
        patience_count = 0
        torch.save(model.state_dict(), CKPT_DIR / 'best.pt')
        print(f'  ↑ New best')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nBest val Dice: {best_dice:.4f}')

## 6. Full Evaluation — Seen + Unseen (Generalization Study)

In [ ]:
# Load best checkpoint and evaluate all splits
model.load_state_dict(torch.load(CKPT_DIR / 'best.pt', map_location=device))
model.eval()
val_transform = get_val_transform(IMG_SIZE)

split_labels = {
    'Seen — Kvasir':        'seen_kvasir',
    'Seen — CVC-ClinicDB':  'seen_clinicdb',
    'Unseen — CVC-ColonDB': 'cvc_colondb',
    'Unseen — ETIS-Larib':  'etis_larib',
    'Unseen — CVC-300':     'cvc_300',
}

print(f'{"Split":<26} {"mDice":>7} {"mIoU":>7} {"MAE":>7} {"wFm":>7} {"Sm":>7} {"Em":>7}')
print('-' * 75)

results = {}
for label, key in split_labels.items():
    if key not in splits:
        print(f'{label:<26} -- not downloaded yet --')
        continue
    ds = PolypDataset(splits[key]['image_paths'], splits[key]['mask_paths'], transform=val_transform)
    loader = DataLoader(ds, batch_size=4, shuffle=False, num_workers=2)
    tracker = MetricTracker()
    with torch.no_grad():
        for imgs, masks in loader:
            probs = torch.sigmoid(model(imgs.to(device))).cpu().numpy()
            for i in range(len(probs)):
                tracker.update(probs[i,0], masks[i,0].numpy())
    sc = tracker.compute()
    results[key] = sc
    print(f'{label:<26} {sc["dice"]:>7.4f} {sc["iou"]:>7.4f} {sc["mae"]:>7.4f} '
          f'{sc["wfm"]:>7.4f} {sc["sm"]:>7.4f} {sc["em"]:>7.4f}')

## 7. Ablation: LoRA rank r

Compare r = {1, 2, 4, 8, 16} on seen Kvasir to choose the best rank.

In [ ]:
# TODO (Phase 4): Run rank ablation sweep
# ranks = [1, 2, 4, 8, 16]
# for r in ranks:
#     model_r = build_sam_lora(fname, SAM_TYPE, lora_r=r, ...)
#     ... train for N epochs ...
#     ... record val Dice ...
print('Rank ablation: scheduled for Phase 4 (Weeks 9-10)')

## Phase 3 Summary

| Deliverable | Status |
|---|---|
| SAM LoRA architecture defined | ✅ |
| Parameter efficiency verified | ✅ |
| Training loop (same protocol as U-Net) | ✅ |
| Full seen/unseen evaluation | ✅ (runs after training) |
| Rank ablation | ⬜ Phase 4 |

**Next:** `04_benchmark.ipynb` (Phase 4) — side-by-side comparison table and generalization analysis.